# convtranspose-bn-activation-block — ex2: ConvT+BN+ReLU 4→9 spatial via output_padding (odd-target upsample)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `convtranspose-bn-activation-block`. Running the final beacon cell reports progress against the `GAN: ConvT+BN+Activation block` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
import torch.nn as nn
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: ConvT+BN+Activation block` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`convtranspose-bn-activation-block`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "convtranspose-bn-activation-block"
DD_SUBTOPIC = "GAN: ConvT+BN+Activation block"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ConvTranspose2d + BN + ReLU — `output_padding` for odd outputs

Ex1 built a `ConvTranspose2d → BN → ReLU` block. The deepening move exercises `output_padding`, the knob that disambiguates the inverse of strided Conv.

**Spatial formula:** `H_out = (H_in - 1)*stride - 2*pad + kernel + output_padding`. 
Forward strided Conv collapses multiple inputs to one output, so the inverse is many-to-one — the transpose needs an extra hint to pick the right output size.

**4 → 9 example.** `H_in=4, stride=2, kernel=3, pad=1`:
- `(4 - 1)*2 - 2*1 + 3 = 6 - 2 + 3 = 7` with `output_padding=0`.
- Add `output_padding=2` → `7 + 2 = 9`. (Constraint: `output_padding < stride OR output_padding < dilation`; `2 < stride? no` — PyTorch actually requires `output_padding < max(stride, dilation)`, and `stride=2` gives an upper bound of 2, so `output_padding=2` is REJECTED.) The correct ARENA pattern picks `output_padding=1, stride=2, kernel=3, pad=1` → `H_out=8` from `H_in=4`, then a second block handles the next jump.

**Pragmatic 4 → 9 path.** Use `stride=2, kernel=4, pad=0, output_padding=1`: `(4-1)*2 - 0 + 4 + 1 = 6 + 4 + 1 = 11`. Doesn't land on 9. Use `stride=2, kernel=3, pad=2, output_padding=1`: `(4-1)*2 - 4 + 3 + 1 = 6 - 4 + 3 + 1 = 6`. Still not 9.

**The clean 4 → 9 spec.** `stride=3, kernel=3, pad=1, output_padding=2`: `(4-1)*3 - 2 + 3 + 2 = 9 - 2 + 3 + 2 = 12`. Closer but still off. The TRUE working spec: `stride=2, kernel=4, pad=1, output_padding=1`: `(4-1)*2 - 2 + 4 + 1 = 6 - 2 + 4 + 1 = 9`. ✓ — and `output_padding=1 < stride=2` satisfies the constraint.

```python
block = nn.Sequential(
    nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4,
                       stride=2, padding=1, output_padding=1, bias=False),
    nn.BatchNorm2d(out_ch),
    nn.ReLU(inplace=True),
)
```

**Why this matters.** Generator stacks frequently need to land on an odd target size (e.g. 7×7 MNIST). `output_padding` is the only non-fractional knob that hits odd outputs from even inputs without switching to fractional strides.

### Exercise 2 — ConvT+BN+ReLU 4→9 spatial via output_padding (odd-target upsample)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `ConvTranspose2d(stride=2, kernel_size=4, padding=1, output_padding=1)` to upsample a 4×4 feature map to 9×9 — exercising the `output_padding` knob that disambiguates the inverse of strided Conv for odd-target outputs.
> Keywords: conv-transpose, output-padding, upsample, odd-output
> ```

**KCs targeted:** `convt-output-padding-formula`, `convt-bn-relu-bias-false`

Build `ex2_build_convt_outpad_block(in_ch, out_ch)`. The odd-target upsample variant of ex1's ConvT block.

Spatial spec: takes `(B, in_ch, 4, 4)` → `(B, out_ch, 9, 9)`.

Derivation. The PyTorch formula is:
```
H_out = (H_in - 1)*stride - 2*padding + kernel + output_padding
```
With `H_in=4, stride=2, kernel=4, padding=1, output_padding=1`:
```
H_out = 3*2 - 2*1 + 4 + 1 = 6 - 2 + 4 + 1 = 9   ✓
```
Constraint check: `output_padding < max(stride, dilation)`. Here `1 < 2`, so this is legal.

Constraints:
1. Return an `nn.Sequential` with three children in order:
   - `nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1, output_padding=1, bias=False)`
   - `nn.BatchNorm2d(out_ch)`
   - `nn.ReLU(inplace=True)`
2. `bias=False` (BN's affine handles the bias).
3. Output H/W exactly `9, 9` from input `4, 4`.

Output: an `nn.Sequential` callable as `block(x)` where `x: (B, in_ch, 4, 4)` → `out: (B, out_ch, 9, 9)`.

In [ ]:
def ex2_build_convt_outpad_block(in_ch, out_ch):
    return nn.Sequential(
        nn.ConvTranspose2d(
            in_ch, out_ch,
            kernel_size=4, stride=2, padding=1, output_padding=1,
            bias=False,
        ),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


<details><summary>Solution</summary>

```python
def ex2_build_convt_outpad_block(in_ch, out_ch):
    return nn.Sequential(
        nn.ConvTranspose2d(
            in_ch, out_ch,
            kernel_size=4, stride=2, padding=1, output_padding=1,
            bias=False,
        ),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )
```

**The 4→9 spec uses `kernel=4, stride=2, pad=1, out_pad=1`.** Walk the formula: `(4-1)*2 - 2 + 4 + 1 = 9`. The `output_padding=1` is what makes this land on an odd target — without it you'd be stuck at 8. PyTorch requires `output_padding < max(stride, dilation)`, so the upper bound on `out_pad` here is `1` (since `stride=2, dilation=1` → max=2, strict less-than gives ≤1).

**Why odd targets matter.** MNIST is 28×28, CelebA-cropped is 64×64, but ImageNet-class outputs and some intermediate feature maps in custom generators land on 7×7, 11×11, etc. `output_padding` is the canonical knob; the alternative is fractional strides via `Upsample` + standard Conv, which have their own gotchas (checkerboard artifacts).

**`bias=False` again.** Same reasoning as the discriminator block — BN absorbs any constant offset, so the ConvT bias would just waste parameters.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()